# style LoRA ablation

compare (base motion module) vs (motion + anime LoRA) vs (motion + cinematic LoRA) on 12 held-out prompts.

we look at FVD, CLIP-T, motion score, and visual samples.

In [ ]:
import os, itertools
import numpy as np
import imageio
from src.inference.t2v import run as run_single
from src.eval.fvd_score import compute_fvd, I3DFeatures
from src.eval.clipsim_temporal import CLIPTemporal
from src.eval.motion_score import motion_score


In [ ]:
PROMPTS = [
    'a fox running through autumn leaves',
    'a spaceship landing on a snowy planet',
    'a chef flipping pancakes in a busy kitchen',
    'city street at night in rain, neon reflections',
    'child on a swing, sunny park',
    'waves crashing on a rocky beach',
    'astronaut planting a flag on mars',
    'dragon flying over a mountain',
    'coffee being poured in slow motion',
    'origami crane unfolding',
    'timelapse of a flower blooming',
    'a corgi doing zoomies in a backyard'
]
STYLES = [None, 'anime', 'cinematic']


In [ ]:
os.makedirs('runs/ablate', exist_ok=True)
for i, p in enumerate(PROMPTS):
    for s in STYLES:
        out = f'runs/ablate/p{i:02d}_{s or "base"}.mp4'
        run_single(prompt=p, out_path=out, style=s, seed=42, motion_scale=1.0)


In [ ]:
# aggregate metrics per style
clip = CLIPTemporal()
rows = []
for s in STYLES:
    ct_means, ms_means = [], []
    for i, p in enumerate(PROMPTS):
        f = np.stack(list(imageio.get_reader(f'runs/ablate/p{i:02d}_{s or "base"}.mp4')))
        ct_means.append(clip.score(p, f)['mean'])
        ms_means.append(motion_score(f)['mean'])
    rows.append({'style': s or 'base', 'clipT': np.mean(ct_means), 'motion': np.mean(ms_means)})
import pandas as pd; pd.DataFrame(rows)
